[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/toche7/SlideAIDATADGA/blob/main/slides/workshop-07-llm-data-extraction-groq.ipynb)

# Workshop 07: LLM for Data Extraction (Groq + Colab)

Notebook นี้สาธิตตามสไลด์หัวข้อ **LLM for Data Extraction**:
1. เชื่อมต่อ LLM (Groq API)
2. ดึงข้อมูลแบบมีโครงสร้างจากข้อความที่ไม่มีโครงสร้าง
3. วิเคราะห์อารมณ์ (Sentiment Analysis)

## 1) Install Required Libraries

In [ ]:
!pip -q install openai pydantic pandas pypdf

## 2) Set API Key and Create Client
สมัคร API key ที่ https://console.groq.com/keys แล้ววางค่าในช่อง input

In [ ]:
import os
from getpass import getpass
from openai import OpenAI

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter GROQ_API_KEY: ")

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "openai/gpt-oss-20b"
print("Client ready. Model:", MODEL)

## 3) Data Extraction from Unstructured Text
ตัวอย่างนี้จะสกัดข้อมูลคำร้องเรียนบริการภาครัฐให้เป็น JSON ที่ใช้งานต่อได้

In [ ]:
from pydantic import BaseModel
from typing import List
import json

class ComplaintRecord(BaseModel):
    citizen_name: str
    province: str
    service_type: str
    waiting_time_minutes: int
    satisfaction_score_1_to_5: int
    issue_summary: str

raw_text = """
รายงานรับฟังความคิดเห็นประชาชน (มีนาคม 2569)
1) นายสมชาย ใจดี จังหวัดเชียงใหม่ มาติดต่อทำบัตรประชาชน ใช้เวลารอ 95 นาที
   ให้คะแนนความพึงพอใจ 2/5 เนื่องจากคิวล่าช้าและข้อมูลเอกสารไม่ชัดเจน
2) นางสาวพิมพ์ชนก ศรีสุข จังหวัดขอนแก่น มาติดต่อชำระภาษีที่ดิน ใช้เวลารอ 25 นาที
   ให้คะแนนความพึงพอใจ 5/5 เจ้าหน้าที่ให้คำแนะนำดีมาก
3) นายอาทิตย์ แสงทอง จังหวัดกรุงเทพมหานคร มาต่อใบอนุญาต ใช้เวลารอ 40 นาที
   ให้คะแนนความพึงพอใจ 3/5 ระบบจองคิวยังใช้งานยาก
"""

prompt = f"""
Extract complaint records from the Thai text below.
Return ONLY valid JSON array with keys exactly:
citizen_name, province, service_type, waiting_time_minutes, satisfaction_score_1_to_5, issue_summary

Text:
{raw_text}
"""

response = client.chat.completions.create(
    model=MODEL,
    temperature=0,
    messages=[
        {"role": "system", "content": "You are an accurate information extraction assistant."},
        {"role": "user", "content": prompt},
    ],
)

json_text = response.choices[0].message.content.strip()
records = json.loads(json_text)

validated = [ComplaintRecord(**r).model_dump() for r in records]
print(json.dumps(validated, ensure_ascii=False, indent=2))

In [ ]:
import pandas as pd

df = pd.DataFrame(validated)
df

## 4) (Optional) Extract Text from PDF then Send to LLM
ใส่ไฟล์ PDF ใน Colab แล้วรันเพื่อดึงข้อความก่อนส่งเข้า LLM

In [ ]:
# Uncomment to upload PDF in Colab
# from google.colab import files
# uploaded = files.upload()
# pdf_path = next(iter(uploaded.keys()))

# from pypdf import PdfReader
# reader = PdfReader(pdf_path)
# pdf_text = "\n".join(page.extract_text() or "" for page in reader.pages)
# print(pdf_text[:2000])

## 5) Sentiment Analysis with LLM
จัดกลุ่มเป็น positive / neutral / negative พร้อมเหตุผลสั้น ๆ

In [ ]:
sentiment_prompt = f"""
Analyze each issue_summary below and return ONLY valid JSON array.
Each item must have keys: issue_summary, sentiment_label, reason_th
Allowed sentiment_label: positive, neutral, negative

Input records:
{json.dumps(validated, ensure_ascii=False)}
"""

sentiment_resp = client.chat.completions.create(
    model=MODEL,
    temperature=0,
    messages=[
        {"role": "system", "content": "You are a strict JSON sentiment classifier."},
        {"role": "user", "content": sentiment_prompt},
    ],
)

sentiments = json.loads(sentiment_resp.choices[0].message.content.strip())
sent_df = pd.DataFrame(sentiments)
sent_df

In [ ]:
# Summary for presentation
summary = sent_df["sentiment_label"].value_counts().rename_axis("sentiment").reset_index(name="count")
summary

## 6) Prompt Templates (for teaching)

**Extraction Prompt**
```
Extract structured data from this text and return JSON only with fields:
name, province, service_type, waiting_time_minutes, satisfaction_score, issue_summary
```

**Sentiment Prompt**
```
Classify sentiment (positive/neutral/negative) for each complaint and return JSON only.
Include a short Thai reason in each record.
```